# envoi walkthrough

envoi allows users to obtain a comprehensive set of environmental features for a set of user-specified geographic locations, in an easy to use Python interface. This notebook serves as a high-level introduction to new users of the API, including key methods and structure of data inputs and outputs.

The library currently supports data extraction and processing from datasets stored on [Google Earth Engine](https://earthengine.google.com/) (GEE) and from raster data stored locally.

Run the cells top to bottom, starting with the Setup section below to install envoi and authenticate with GEE. Inputs live in [walkthrough_input/](walkthrough_input/); outputs are written to `walkthrough_output/` next to this notebook.

1. Tabular extraction from a built-in GEE dataset
2. Raster tile extraction (GeoTIFFs per point)
3. Registering and extracting from a local raster
4. Multiple datasets and mixed continuous/categorical reducers in one call
5. Date-aware extraction with `eventDate`
6. Discovering what's in the catalog

## Setup

Install envoi (Python 3.10+):

```bash
pip install envoi #CHANGE TO CORRECT BEFORE RELEASE
```

Earth Engine sections need a Google service-account JSON key. See the [README](../README.md#earth-engine-setup) for how to obtain a key. If you only want to try the local-raster section, skip the `init_gee()` cell.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import rasterio

from envoi import extract, init_gee, list_datasets, list_reducers, update_catalog

OUTPUT_DIR = Path("walkthrough_output")

In [ ]:
init_gee()

## Input data

envoi expects a `DataFrame` of sample points. The defaults follow the GBIF / Darwin Core convention:

| column              | required | description                              |
| ------------------- | -------- | ---------------------------------------- |
| `gbifID`            | yes      | row identifier                           |
| `decimalLatitude`   | yes      | latitude in degrees                      |
| `decimalLongitude`  | yes      | longitude in degrees                     |
| `eventDate`         | no       | `YYYY-MM-DD` — used for time-varying datasets |

Coordinates are assumed to be in **WGS84 (EPSG:4326)**. If yours are in a different CRS, pass `input_crs="EPSG:XXXX"` to `extract()`. If your columns are named differently (e.g. `id`/`lat`/`lon`), pass `id_column=`, `latitude_column=`, etc.

The five points below are five occurrences of *Linnaea borealis* in an area outside of Uppsala, Sweden, downloaded from GBIF.

In [ ]:
gbif_points = pd.read_csv("walkthrough_input/sample_points.csv", delimiter="\t")
gbif_points

Let's condense the dataframe to the columns needed for this walkthrough

In [ ]:
points = gbif_points[['gbifID', 'decimalLatitude', 'decimalLongitude', 'eventDate']]
points

## 1. Tabular extraction from Earth Engine

In this example, we fetch elevation data from the [Copernicus DEM GLO-30](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_DEM_GLO30), a global digital elevation model with roughly 30-metre pixel resolution. Elevation is one of the most commonly used predictors in species distribution modelling — it correlates with temperature, precipitation, and vegetation structure — so it serves as a clear and familiar starting point.

For each point, envoi extracts all pixels within a 500-metre square window centred on the coordinate and computes summary statistics across them. Here we request the mean (average elevation) and standard deviation (a proxy for terrain roughness) within each window. Setting `output_type: "tabular"` produces three files in `OUTPUT_DIR`:

- `terrain_stats.csv` — one row per input point, one column per statistic
- `terrain_stats_qc.csv` — per-point quality control flags (covered below)
- `terrain_stats_metadata.json` — run settings and dataset provenance

Output columns are named `<dataset>_<statistic>_<window>m`, so `dem_copernicus_glo30_mean_500m` is the mean elevation within the 500-metre window.

In [ ]:
extract(
    points,
    {
        "batch_id": "terrain_stats",
        "datasets": ["dem_copernicus_glo30"],
        "settings": {
            "output_type": "tabular",
            "statistics": ["mean", "std"],
            "window_size_m": 500,
        },
    },
    output_dir=OUTPUT_DIR,
)

pd.read_csv(OUTPUT_DIR / "terrain_stats.csv")

The call to `extract()` also generates a CSV with quality control (QC) metrics that can be useful to diagnose potential issues:

- `in_extent (bool)`: whether the sample location or sampling window overlaps the available data source.
- `n_pixels (int)`: the number of pixels included in the sampled result.
- `had_nodata (bool)`: whether nodata or missing values were encountered inside the sampled window.
- `coverage_pct (float)`: the percentage of sampled pixels that contained valid data.
- `image_time_start (str)`: for temporal GEE collections, the start date of the image actually used for that sample (YYYY-MM-DD).
- `image_time_end (str)`: end date of the image used, when available.
- `date_clamped (bool)`: whether the requested sample date had to be clamped to the nearest available date in the collection.
- `date_source (str)`: how the image date was chosen, for example `nearest_to_sample`, `clamped_to_nearest`, or `most_recent_no_date`.
- `region_crs (str)`: the CRS used for the sampled region or exported tile geometry.

In [ ]:
pd.read_csv(OUTPUT_DIR / "terrain_stats_qc.csv")

The `_metadata.json` sidecar records everything needed to reproduce or audit a run: the timestamp and package version (`run`) and the effective settings used (`config`). The `datasets` section holds per-dataset source details — native CRS, resolution, band names, coverage counts, and for temporal GEE collections a summary of which images were selected. A `warnings` key is added when any inputs were adjusted, such as dates outside the collection range or coordinates reprojected to WGS84.

In [ ]:
metadata = json.loads((OUTPUT_DIR / "terrain_stats_metadata.json").read_text())
print(json.dumps(metadata, indent=2))

## 2. Raster tile extraction

Switching `output_type` to `"raster"` exports one GeoTIFF image tile per point, rather than summarising the pixels into a single number. Each tile is a spatial crop of the source data centred on the sampling location — useful when you need the full spatial pattern within the window, for example to compute custom landscape metrics, feed into convolutional neural networks, or visually inspect the terrain context around each occurrence.

Setting `resample_m: 10` reprojects every tile to the point's local UTM zone at a fixed 10-metre pixel size, so tiles from different points and datasets share the same dimensions and can be compared or stacked directly. With a 500-metre window and 10 m/pixel that gives a 50 × 50 pixel tile per point. Omit `resample_m` to keep the native resolution of the source dataset; the tile dimensions are rounded to the nearest whole pixel count, so the actual covered area may differ slightly from the requested window size (e.g. a 500-metre window at 30 m/pixel becomes 17 pixels, covering 510 metres).

Tiles are saved to `OUTPUT_DIR/<batch_id>/<dataset>/<id>-<dataset>.tif` and metadata to `OUTPUT_DIR/<batch_id>_metadata.json`

In [ ]:
extract(
    points,
    {
        "batch_id": "terrain_tiles",
        "datasets": ["dem_copernicus_glo30"],
        "settings": {
            "output_type": "raster",
            "window_size_m": 500,
            "resample_m": 10,
        },
    },
    output_dir=OUTPUT_DIR,
)

tiles = sorted((OUTPUT_DIR / "terrain_tiles" / "dem_copernicus_glo30").glob("*.tif"))
tiles

We can view the pixel-wise elevation values of one of the extracted GeoTIFF images below:

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
with rasterio.open(tiles[0]) as src:
    im = ax.imshow(src.read(1), cmap="Greys")
ax.set_title(tiles[0].stem.split("-")[0], fontsize=9)
ax.set_axis_off()
fig.colorbar(im, ax=ax, label="Elevation (m)", shrink=0.8)
fig.suptitle("dem_copernicus_glo30")
plt.tight_layout()
plt.show()

## 3. Local rasters

Not all useful datasets are available on Google Earth Engine. High-resolution national maps, habitat models from conservation organisations, custom-processed remote sensing products, or datasets shared by collaborators often exist only as local raster files on your machine. envoi can work with these in exactly the same way as built-in GEE data — register the file once via `update_catalog()`, and then reference it by name in any `extract()` call.

CRS, pixel size, nodata value, and band count are read automatically from the file metadata, so there is no need to specify them manually. Below we register the `biomass.tif`, a small GeoTIFF of a national product ([SLU Forest Map](https://www.slu.se/en/environment/statistics-and-environmental-data/environmental-data-catalogue/slu-forest-map/)), and then run another extract() call.

In [ ]:
update_catalog({
    "datasets": {
        "biomass_local": {
            "data_source": "local",
            "path": "walkthrough_input/biomass.tif",
        },
    },
})

In [ ]:
extract(
    points,
    {
        "batch_id": "biomass_stats",
        "datasets": ["biomass_local"],
        "settings": {
            "output_type": "tabular",
            "statistics": ["mean", "std"],
            "window_size_m": 200,
        },
    },
    output_dir=OUTPUT_DIR,
)

pd.read_csv(OUTPUT_DIR / "biomass_stats.csv")

## 4. Multiple datasets in one call

Building a useful predictor table for species distribution modelling or similar analyses typically means combining many environmental layers: topography, climate, land cover, vegetation indices, and more. A single `extract()` call handles multiple datasets at once — including a mix of GEE and local rasters — so the entire feature table is assembled in one step for exactly the same set of locations.

When datasets include both continuous variables (like elevation, where mean and standard deviation are meaningful) and categorical variables (like land cover, where each pixel belongs to a discrete class such as forest or wetland), pass `statistics` as a typed dict assigning different reducers to each type. For categorical data, envoi returns the dominant class within the window and the fraction of pixels belonging to each class — for example, the proportion of a 500-metre window covered by forest versus open water. Reducers that apply to both types, such as `mode`, can appear in both lists.

Below we combine the [Copernicus DEM](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_DEM_GLO30) (continuous, GEE), the local biomass tif (continuous, local), and [ESA WorldCover 2021](https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200) (categorical, GEE) in a single call. 
>Note: For the local biomass we did not define the data type (categorical/continuous), so it defaults to continuous.

In [ ]:
extract(
    points,
    {
        "batch_id": "multi_source",
        "datasets": ["dem_copernicus_glo30", "biomass_local", "lulc_worldcover_2021"],
        "settings": {
            "output_type": "tabular",
            "window_size_m": 500,
            "statistics": {
                "continuous": ["mean", "std"],
                "categorical": ["class_fraction"],
            },
        },
    },
    output_dir=OUTPUT_DIR,
)

pd.read_csv(OUTPUT_DIR / "multi_source.csv")

For the lulc_worldcover_2021 (lulc stands for land use / land cover), there are two classes represented within a 500 meter window for 4 of 5 points (class 10 and 30). Let's check the [documentation on GEE](https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200#bands) for these classes represent.  
Class 10 represents tree cover and class 30 represents grasslands. For the first sample (gbifID: 3767408287) there are 94.3% tree coverage and 5.7% grassland within our designated study area.

## 5. Date-aware extraction

Many environmental datasets vary over time — vegetation greenness, snow cover, land surface temperature — and a key feature of envoi is that it can match each sampling point to the environmental conditions at the time of the observation, rather than returning a single time-averaged value. When the input table includes an `eventDate` column and the dataset is a time-varying GEE `ImageCollection`, envoi automatically selects one image per point based on its date.

Dates outside the collection's available range are silently clamped to the nearest boundary and recorded in the QC output (`*_date_clamped_*` and `*_image_time_start_*`). Without an `eventDate` column the most recent available image is used for all points.

Below we extract annual NDVI for each occurrence point, matched to the year of the observation.

In [ ]:
extract(
    points,
    {
        "batch_id": "ndvi",
        "datasets": ["ndvi_landsat_annual"],
        "settings": {
            "output_type": "tabular",
            "statistics": ["mean"],
            "window_size_m": 200,
        },
    },
    output_dir=OUTPUT_DIR,
)

ndvi = pd.read_csv(OUTPUT_DIR / "ndvi.csv")
ndvi_qc = pd.read_csv(OUTPUT_DIR / "ndvi_qc.csv")
ndvi.merge(
    ndvi_qc.filter(regex=r"(gbifID|image_time_start|image_time_end)"),
    on="gbifID",
)

## 6. Discovering what's available

Before building an extraction config it can be useful to browse which datasets are in the catalog and which statistics are supported. `list_datasets()` prints and returns the names of all built-in datasets plus any added via `update_catalog()` in the current session — `biomass_local` appears in the output below because we registered it in Section 3. Pass `"info"` to include short descriptions and citations for each dataset, or `"full"` to see the complete catalog entry including band names, native resolution, and temporal coverage.

`list_reducers()` returns the names of all spatial statistics that can appear in the `statistics` field of a config — a quick reference when setting up a new run.

In [ ]:
list_datasets()

In [ ]:
list_reducers()

## Where to go next

- [README](../README.md) — install, GEE setup, full feature reference
- [docs/advanced_usage.md](../docs/advanced_usage.md) — per-call band selection, custom catalog YAML files, mixed-type runs in more depth
- [examples/run.yml](run.yml) — a runnable YAML config you can pass directly to `extract()` instead of a Python dict
- [examples/catalog.yml](catalog.yml) — a starter custom-catalog template for `update_catalog()`